# StayNest - Session 6 Assignment (PySpark Deep Dive)
Work through the 8 tasks below in order. Read the Assignment Questions PDF for the
full detail and acceptance criteria. Fill in each `# TODO` cell, run it, and keep the
output visible. Run on Databricks Free Edition (serverless).

## Section 0 - Setup (already done for you)
Upload `bookings.csv`, `hotels.csv`, `customers.csv` to a Volume, then set `BASE`
to that path and run this cell. Counts should be 12000 / 200 / 2000.

In [0]:
# Point BASE at YOUR Volume path
BASE = "/Volumes/workspace/default/staynest"

print(spark.version)

read_csv = lambda name: (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{BASE}/{name}.csv"))

bookings_df   = read_csv("bookings")
hotels_df     = read_csv("hotels")
customers_df  = read_csv("customers")

print(f"bookings: {bookings_df.count()}, "
      f"hotels: {hotels_df.count()}, "
      f"customers: {customers_df.count()}")

4.1.0
bookings: 12000, hotels: 200, customers: 2000


## Task 1 - Read and inspect
Show the schema, a few sample rows, the row count, and summary stats for the
numeric columns of `bookings_df`.

In [0]:
from pyspark.sql.types import NumericType

# 1. Show the DataFrame schema
bookings_df.printSchema()

# 2. Show five sample rows
bookings_df.show(5, truncate=False)

# 3. Show the total row count
bookings_count = bookings_df.count()
print(f"Total bookings: {bookings_count}")

# 4. Identify all numeric columns from the schema
numeric_columns = [
    field.name
    for field in bookings_df.schema.fields
    if isinstance(field.dataType, NumericType)
]

print(f"Numeric columns: {numeric_columns}")

# 5. Show summary statistics for all numeric columns
bookings_df.select(*numeric_columns).describe().show()

root
 |-- booking_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- hotel_id: integer (nullable = true)
 |-- booking_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- nights: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)

+----------+-----------+--------+------------+---------+------+--------+---------+
|booking_id|customer_id|hotel_id|booking_date|city     |nights|amount  |status   |
+----------+-----------+--------+------------+---------+------+--------+---------+
|9000000   |701600     |3095    |2025-11-27  |Jaipur   |4     |6087.65 |completed|
|9000001   |700065     |3057    |2025-11-06  |Delhi    |1     |8211.19 |cancelled|
|9000002   |701392     |3187    |2025-08-21  |Jaipur   |2     |7176.52 |cancelled|
|9000003   |700867     |3112    |2025-03-22  |Bengaluru|5     |7880.62 |completed|
|9000004   |701521     |3043    |2025-04-19  |Mumbai   |5     |21021.51|pending  |
+--------

## Task 2 - Select and filter
From `bookings_df`, select a few useful columns and return the **completed**
bookings with `amount` over 10000 in the cities Goa or Mumbai. Use `col()`, combine
conditions with `&`, and use `.isin(...)`.

In [0]:
from pyspark.sql.functions import col

completed_high_value_bookings = (
    bookings_df
    .select(
        col("booking_id"),
        col("customer_id"),
        col("hotel_id"),
        col("booking_date"),
        col("city"),
        col("nights"),
        col("amount"),
        col("status")
    )
    .filter(
        (col("status") == "completed")
        & (col("amount") > 10000)
        & (col("city").isin("Goa", "Mumbai"))
    )
)

completed_high_value_bookings.show(truncate=False)


+----------+-----------+--------+------------+------+------+--------+---------+
|booking_id|customer_id|hotel_id|booking_date|city  |nights|amount  |status   |
+----------+-----------+--------+------------+------+------+--------+---------+
|9000007   |700868     |3127    |2025-04-10  |Mumbai|2     |15693.64|completed|
|9000010   |700174     |3054    |2025-10-19  |Goa   |6     |30786.36|completed|
|9000028   |701748     |3022    |2025-02-07  |Goa   |3     |27008.87|completed|
|9000034   |700110     |3118    |2025-04-19  |Goa   |7     |64388.77|completed|
|9000050   |701894     |3147    |2025-02-20  |Mumbai|6     |24275.14|completed|
|9000061   |701454     |3115    |2025-03-06  |Mumbai|5     |19180.93|completed|
|9000070   |701629     |3118    |2025-01-12  |Goa   |7     |58790.91|completed|
|9000071   |700768     |3159    |2025-06-10  |Mumbai|6     |47548.35|completed|
|9000075   |700312     |3006    |2025-04-21  |Goa   |5     |18043.52|completed|
|9000087   |700953     |3099    |2025-10

## Task 3 - Derived columns
Add: `amount_with_gst` (amount plus 12% tax), a `value_tier`
(premium / standard / budget) using `when`/`otherwise`, and a `booking_month`
from `booking_date`.

In [0]:
from pyspark.sql.functions import col, when, date_format, to_date
from pyspark.sql.functions import round as spark_round

bookings_enriched_df = (
    bookings_df
    .withColumn(
        "amount_with_gst",
        spark_round(col("amount") * 1.12, 2)
    )
    .withColumn(
        "value_tier",
        when(col("amount") >= 20000, "premium")
        .when(col("amount") >= 10000, "standard")
        .otherwise("budget")
    )
    .withColumn(
        "booking_month",
        date_format(to_date(col("booking_date")), "yyyy-MM")
    )
)

# Show a sample containing the original and derived values
bookings_enriched_df.select(
    "booking_id",
    "booking_date",
    "amount",
    "amount_with_gst",
    "value_tier",
    "booking_month"
).show(10, truncate=False)


+----------+------------+--------+---------------+----------+-------------+
|booking_id|booking_date|amount  |amount_with_gst|value_tier|booking_month|
+----------+------------+--------+---------------+----------+-------------+
|9000000   |2025-11-27  |6087.65 |6818.17        |budget    |2025-11      |
|9000001   |2025-11-06  |8211.19 |9196.53        |budget    |2025-11      |
|9000002   |2025-08-21  |7176.52 |8037.7         |budget    |2025-08      |
|9000003   |2025-03-22  |7880.62 |8826.29        |budget    |2025-03      |
|9000004   |2025-04-19  |21021.51|23544.09       |premium   |2025-04      |
|9000005   |2025-10-14  |18124.49|20299.43       |standard  |2025-10      |
|9000006   |2025-11-25  |70999.15|79519.05       |premium   |2025-11      |
|9000007   |2025-04-10  |15693.64|17576.88       |standard  |2025-04      |
|9000008   |2025-01-15  |1492.62 |1671.73        |budget    |2025-01      |
|9000009   |2025-06-08  |6694.39 |7497.72        |budget    |2025-06      |
+----------+

## Task 4 - Aggregations
For **completed** bookings, group by `city` and return: number of bookings, total
revenue, average amount, biggest booking, and the count of unique customers.
Order by revenue, highest first.

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    sum as spark_sum,
    avg,
    max as spark_max,
    countDistinct
)

city_revenue_df = (
    bookings_df
    .filter(col("status") == "completed")
    .groupBy("city")
    .agg(
        count("booking_id").alias("booking_count"),
        spark_sum("amount").alias("total_revenue"),
        avg("amount").alias("average_amount"),
        spark_max("amount").alias("biggest_booking"),
        countDistinct("customer_id").alias("unique_customers")
    )
    .orderBy(col("total_revenue").desc())
)

city_revenue_df.show(truncate=False)


+---------+-------------+--------------------+------------------+---------------+----------------+
|city     |booking_count|total_revenue       |average_amount    |biggest_booking|unique_customers|
+---------+-------------+--------------------+------------------+---------------+----------------+
|Goa      |2546         |4.459670178999999E7 |17516.379336213664|78481.24       |1441            |
|Mumbai   |1715         |3.624122112E7       |21131.90735860058 |78524.52       |1153            |
|Delhi    |1174         |2.631428154000001E7 |22414.209148211252|78669.69       |861             |
|Jaipur   |979          |2.4436853129999984E7|24961.03486210417 |76904.87       |796             |
|Bengaluru|1318         |2.267013697000002E7 |17200.4074127466  |78568.81       |969             |
|Udaipur  |691          |1.2094427419999994E7|17502.78931982633 |77939.5        |592             |
|Rishikesh|407          |8606121.57999999    |21145.261867321846|77458.26       |363             |
|Manali   

## Task 5 - Joins
Inner-join bookings to hotels to enrich each booking. Do a left join too. Use
`left_anti` to check for orphaned bookings (expect 0). Then do a three-way join
with customers.

In [0]:
from pyspark.sql.functions import col

# Aliases prevent ambiguity because bookings, hotels, and customers
# all contain a column named city.
b = bookings_df.alias("b")
h = hotels_df.alias("h")
c = customers_df.alias("c")


# 1. Inner join: keep bookings that have a matching hotel
bookings_hotels_inner_df = (
    b.join(
        h,
        col("b.hotel_id") == col("h.hotel_id"),
        how="inner"
    )
    .select(
        col("b.*"),
        col("h.hotel_name"),
        col("h.city").alias("hotel_city"),
        col("h.category"),
        col("h.star_rating")
    )
)

print("Inner join sample:")
bookings_hotels_inner_df.show(5, truncate=False)


# 2. Left join: preserve every booking even if no hotel matches
bookings_hotels_left_df = (
    b.join(
        h,
        col("b.hotel_id") == col("h.hotel_id"),
        how="left"
    )
    .select(
        col("b.*"),
        col("h.hotel_name"),
        col("h.city").alias("hotel_city"),
        col("h.category"),
        col("h.star_rating")
    )
)

print(f"Left join row count: {bookings_hotels_left_df.count()}")
bookings_hotels_left_df.show(5, truncate=False)


# 3. Left anti join: bookings whose hotel_id does not exist in hotels
orphaned_bookings_df = (
    b.join(
        h,
        col("b.hotel_id") == col("h.hotel_id"),
        how="left_anti"
    )
)

orphan_count = orphaned_bookings_df.count()
print(f"Orphaned bookings: {orphan_count}")


# 4. Three-way join: bookings + hotels + customers
bookings_full_enriched_df = (
    b.join(
        h,
        col("b.hotel_id") == col("h.hotel_id"),
        how="inner"
    )
    .join(
        c,
        col("b.customer_id") == col("c.customer_id"),
        how="inner"
    )
    .select(
        col("b.booking_id"),
        col("b.booking_date"),
        col("b.customer_id"),
        col("c.customer_name"),
        col("c.membership"),
        col("c.city").alias("customer_city"),
        col("b.hotel_id"),
        col("h.hotel_name"),
        col("h.city").alias("hotel_city"),
        col("h.category"),
        col("h.star_rating"),
        col("b.city").alias("booking_city"),
        col("b.nights"),
        col("b.amount"),
        col("b.status")
    )
)

print("Three-way join sample:")
bookings_full_enriched_df.show(5, truncate=False)


Inner join sample:
+----------+-----------+--------+------------+---------+------+--------+---------+--------------+----------+--------+-----------+
|booking_id|customer_id|hotel_id|booking_date|city     |nights|amount  |status   |hotel_name    |hotel_city|category|star_rating|
+----------+-----------+--------+------------+---------+------+--------+---------+--------------+----------+--------+-----------+
|9000000   |701600     |3095    |2025-11-27  |Jaipur   |4     |6087.65 |completed|Orchid Suites |Jaipur    |Budget  |3.8        |
|9000001   |700065     |3057    |2025-11-06  |Delhi    |1     |8211.19 |cancelled|Orchid Retreat|Delhi     |Luxury  |4.1        |
|9000002   |701392     |3187    |2025-08-21  |Jaipur   |2     |7176.52 |cancelled|Marigold Stay |Jaipur    |Premium |3.8        |
|9000003   |700867     |3112    |2025-03-22  |Bengaluru|5     |7880.62 |completed|Grand Stay    |Bengaluru |Budget  |3.6        |
|9000004   |701521     |3043    |2025-04-19  |Mumbai   |5     |21021.51

## Task 6 - Spark SQL + a window function
Register temp views and use `spark.sql` to get revenue by hotel `category` for
completed bookings. Then use a window function to rank the **top 3 hotels by
revenue within each city**.

In [0]:
# Register the DataFrames as temporary SQL views
bookings_df.createOrReplaceTempView("bookings")
hotels_df.createOrReplaceTempView("hotels")

category_revenue_df = spark.sql("""
    SELECT
        h.category,
        COUNT(b.booking_id) AS booking_count,
        ROUND(SUM(b.amount), 2) AS total_revenue,
        ROUND(AVG(b.amount), 2) AS average_booking_amount
    FROM bookings b
    INNER JOIN hotels h
        ON b.hotel_id = h.hotel_id
    WHERE b.status = 'completed'
    GROUP BY h.category
    ORDER BY total_revenue DESC
""")

category_revenue_df.show(truncate=False)

+--------+-------------+-------------+----------------------+
|category|booking_count|total_revenue|average_booking_amount|
+--------+-------------+-------------+----------------------+
|Luxury  |2818         |1.068376269E8|37912.57              |
|Premium |3335         |5.825564086E7|17467.96              |
|Budget  |3507         |2.233825323E7|6369.62               |
+--------+-------------+-------------+----------------------+



In [0]:
top_3_hotels_by_city_df = spark.sql("""
    WITH hotel_revenue AS (
        SELECT
            h.city,
            h.hotel_id,
            h.hotel_name,
            h.category,
            ROUND(SUM(b.amount), 2) AS total_revenue
        FROM bookings b
        INNER JOIN hotels h
            ON b.hotel_id = h.hotel_id
        WHERE b.status = 'completed'
        GROUP BY
            h.city,
            h.hotel_id,
            h.hotel_name,
            h.category
    ),
    ranked_hotels AS (
        SELECT
            city,
            hotel_id,
            hotel_name,
            category,
            total_revenue,
            ROW_NUMBER() OVER (
                PARTITION BY city
                ORDER BY total_revenue DESC, hotel_id
            ) AS city_rank
        FROM hotel_revenue
    )
    SELECT
        city,
        city_rank,
        hotel_id,
        hotel_name,
        category,
        total_revenue
    FROM ranked_hotels
    WHERE city_rank <= 3
    ORDER BY city, city_rank
""")

top_3_hotels_by_city_df.show(truncate=False)


+---------+---------+--------+----------------+--------+-------------+
|city     |city_rank|hotel_id|hotel_name      |category|total_revenue|
+---------+---------+--------+----------------+--------+-------------+
|Anantapur|1        |3036    |Azure Retreat   |Luxury  |1434328.39   |
|Anantapur|2        |3189    |Lotus Retreat   |Premium |617708.27    |
|Anantapur|3        |3061    |Orchid Residency|Budget  |205043.99    |
|Bengaluru|1        |3003    |Serene Inn      |Luxury  |1633509.09   |
|Bengaluru|2        |3162    |Willow Suites   |Luxury  |1433492.08   |
|Bengaluru|3        |3175    |Heritage Stay   |Luxury  |1400383.85   |
|Delhi    |1        |3057    |Orchid Retreat  |Luxury  |2616612.56   |
|Delhi    |2        |3104    |Marigold Stay   |Luxury  |2327675.63   |
|Delhi    |3        |3012    |Orchid Residency|Luxury  |2264974.65   |
|Goa      |1        |3118    |Palm Retreat    |Luxury  |3112412.42   |
|Goa      |2        |3033    |Lotus Inn       |Luxury  |3030434.53   |
|Goa  

## Task 7 - Write the result
Write your city-revenue result as **Parquet**, and also as a **Delta table** with
`saveAsTable`. Read the Delta table back to confirm.

In [0]:
# Output destinations
PARQUET_PATH = f"{BASE}/outputs/city_revenue_parquet"
DELTA_TABLE = "workspace.default.city_revenue"


# 1. Write the city-revenue result as Parquet files
(
    city_revenue_df.write
    .mode("overwrite")
    .parquet(PARQUET_PATH)
)

print(f"Parquet result written to: {PARQUET_PATH}")


# Read the Parquet result back to confirm the write
city_revenue_parquet_df = spark.read.parquet(PARQUET_PATH)

print("Parquet result read back:")
city_revenue_parquet_df.show(truncate=False)


# 2. Write the same result as a registered Delta table
(
    city_revenue_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DELTA_TABLE)
)

print(f"Delta table written to: {DELTA_TABLE}")


# Read the Delta table back from Unity Catalog
city_revenue_delta_df = spark.table(DELTA_TABLE)

print(f"Delta table exists: {spark.catalog.tableExists(DELTA_TABLE)}")
print("Delta table read back:")
city_revenue_delta_df.show(truncate=False)


Parquet result written to: /Volumes/workspace/default/staynest/outputs/city_revenue_parquet
Parquet result read back:
+---------+-------------+--------------------+------------------+---------------+----------------+
|city     |booking_count|total_revenue       |average_amount    |biggest_booking|unique_customers|
+---------+-------------+--------------------+------------------+---------------+----------------+
|Goa      |2546         |4.459670178999999E7 |17516.379336213664|78481.24       |1441            |
|Mumbai   |1715         |3.624122112E7       |21131.90735860058 |78524.52       |1153            |
|Delhi    |1174         |2.631428154000001E7 |22414.209148211252|78669.69       |861             |
|Jaipur   |979          |2.4436853129999984E7|24961.03486210417 |76904.87       |796             |
|Bengaluru|1318         |2.267013697000002E7 |17200.4074127466  |78568.81       |969             |
|Udaipur  |691          |1.2094427419999994E7|17502.78931982633 |77939.5        |592      

## Task 8 - One chained pipeline
In a single chain: keep completed bookings, join hotels, keep hotels with
`star_rating >= 4.0`, group by `category`, sum revenue, order descending, take the
top 5. End with one `.show()`.

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

(
    bookings_df
    .filter(col("status") == "completed")
    .join(hotels_df, on="hotel_id", how="inner")
    .filter(col("star_rating") >= 4.0)
    .groupBy("category")
    .agg(
        spark_sum("amount").alias("total_revenue")
    )
    .orderBy(col("total_revenue").desc())
    .limit(5)
    .show(truncate=False)
)

+--------+--------------------+
|category|total_revenue       |
+--------+--------------------+
|Luxury  |5.8701580909999914E7|
|Premium |2.2516532060000006E7|
|Budget  |7790394.569999994   |
+--------+--------------------+

